In [ ]:
# BLOK — Segment-level CSV ile tüm korpus PC1/PC5 enerji metrikleri
# Kaynak:
# C:\Users\yigit\Desktop\Jedi Order\enlistment data\us_political_corpus\processed\presidency_segments_emfd_projection.csv
#
# Çıktılar:
# 1. document-level CSV
# 2. speaker-level CSV
# 3. Excel summary

from pathlib import Path
import numpy as np
import pandas as pd


# ===================================================
# 1. Ayarlar
# ===================================================

SEGMENT_CSV_PATH = Path(
    r"C:\Users\yigit\Desktop\Jedi Order\enlistment data\us_political_corpus\processed\presidency_segments_emfd_projection.csv"
)

PCA_EXPORT_PATH = Path(
    r"C:\Users\yigit\Desktop\Jedi Order\enlistment data\us_political_corpus\pca_export\pca_export_summary.xlsx"
)

OUT_DIR = Path("corpus_pc_energy_metrics_segment_level")
OUT_DIR.mkdir(parents=True, exist_ok=True)

EPS = 1e-12

MFD_COLS = [
    "care_p",
    "fairness_p",
    "loyalty_p",
    "authority_p",
    "sanctity_p",
]

USECOLS = [
    "doc_idx",
    "segment_idx",
    "segment_id",
    "speaker",
    "party",
    "title",
    "document_date",
    "document_type",
    "word_count",
    "sentence_count",
    "char_count",
    "care_p",
    "fairness_p",
    "loyalty_p",
    "authority_p",
    "sanctity_p",
    "content",
]


# ===================================================
# 2. PCA export oku
# ===================================================

def load_pca_export(path):
    xls = pd.ExcelFile(path)

    scaler_stats = pd.read_excel(path, sheet_name="scaler_stats")

    if "loading_matrix_aligned" in xls.sheet_names:
        loadings = pd.read_excel(path, sheet_name="loading_matrix_aligned")
    else:
        loadings = pd.read_excel(path, sheet_name="loading_matrix")

    mean = scaler_stats.set_index("feature")["mean"].to_dict()
    scale = scaler_stats.set_index("feature")["scale"].to_dict()

    loading_matrix = {}

    for pc in ["PC1", "PC2", "PC3", "PC4", "PC5"]:
        loading_matrix[pc] = (
            loadings
            .set_index("feature")
            .loc[MFD_COLS, pc]
            .to_numpy(dtype=float)
        )

    return mean, scale, loading_matrix


pca_mean, pca_scale, pca_loadings = load_pca_export(PCA_EXPORT_PATH)

print("PCA export loaded.")
print("PC1 loading:", pca_loadings["PC1"])
print("PC5 loading:", pca_loadings["PC5"])


# ===================================================
# 3. Segment CSV oku
# ===================================================

seg_df = pd.read_csv(
    SEGMENT_CSV_PATH,
    usecols=USECOLS,
)

for c in MFD_COLS:
    seg_df[c] = pd.to_numeric(seg_df[c], errors="coerce").fillna(0)

for c in ["doc_idx", "segment_idx", "word_count", "sentence_count", "char_count"]:
    seg_df[c] = pd.to_numeric(seg_df[c], errors="coerce")

seg_df = seg_df.dropna(subset=["doc_idx"]).copy()
seg_df["doc_idx"] = seg_df["doc_idx"].astype(int)

seg_df["word_count"] = seg_df["word_count"].fillna(0)
seg_df["sentence_count"] = seg_df["sentence_count"].fillna(0)
seg_df["char_count"] = seg_df["char_count"].fillna(0)

print("Segment rows:", len(seg_df))
display(seg_df.head())


# ===================================================
# 3B. JSON metadata ile speaker/party/document_type doldur
# title + date + content_prefix üzerinden eşleştirme
# ===================================================

import json
import re

JSON_PATH = Path(
    r"C:\Users\yigit\Desktop\Jedi Order\enlistment data\us_political_corpus\processed\presidency_enriched_emfd_rhythm.json"
)

def norm_text(x):
    if pd.isna(x):
        return ""
    x = str(x).lower()
    x = re.sub(r"\s+", " ", x)
    return x.strip()

def norm_date(x):
    return pd.to_datetime(x, errors="coerce").strftime("%Y-%m-%d")

def prefix_key(x, n=120):
    return norm_text(x)[:n]

with open(JSON_PATH, "r", encoding="utf-8") as f:
    meta = json.load(f)

json_rows = []

for president_name, pdata in meta.items():
    if not isinstance(pdata, dict):
        continue

    for doc_type_key, docs in pdata.items():
        if not isinstance(docs, list):
            continue

        for doc in docs:
            if not isinstance(doc, dict):
                continue

            json_rows.append({
                "json_president": president_name,
                "json_speaker": doc.get("speaker", president_name),
                "json_party": doc.get("party", None),
                "json_title": doc.get("title", None),
                "json_document_date": doc.get("document_date", None),
                "json_document_type": doc.get("document_type", doc_type_key),
                "json_content_prefix": prefix_key(doc.get("content", ""), 120),
            })

json_meta_df = pd.DataFrame(json_rows)

json_meta_df["merge_title"] = json_meta_df["json_title"].apply(norm_text)
json_meta_df["merge_date"] = pd.to_datetime(
    json_meta_df["json_document_date"],
    errors="coerce"
).dt.strftime("%Y-%m-%d")
json_meta_df["merge_prefix"] = json_meta_df["json_content_prefix"]

# Segment CSV'de her doc için ilk segment content'ini al
doc_first_segment = (
    seg_df
    .sort_values(["doc_idx", "segment_idx"])
    .groupby("doc_idx", as_index=False)
    .agg(
        merge_title=("title", lambda x: norm_text(x.iloc[0])),
        merge_date=("document_date", lambda x: pd.to_datetime(x.iloc[0], errors="coerce").strftime("%Y-%m-%d")),
        merge_prefix=("content", lambda x: prefix_key(x.iloc[0], 120)),
    )
)

json_map = (
    json_meta_df[
        [
            "merge_title",
            "merge_date",
            "merge_prefix",
            "json_president",
            "json_speaker",
            "json_party",
            "json_document_type",
        ]
    ]
    .drop_duplicates(["merge_title", "merge_date", "merge_prefix"])
)

doc_meta_map = doc_first_segment.merge(
    json_map,
    on=["merge_title", "merge_date", "merge_prefix"],
    how="left",
)

print("Doc metadata matched:", doc_meta_map["json_speaker"].notna().sum(), "/", len(doc_meta_map))

seg_df = seg_df.merge(
    doc_meta_map[
        [
            "doc_idx",
            "json_president",
            "json_speaker",
            "json_party",
            "json_document_type",
        ]
    ],
    on="doc_idx",
    how="left",
)

seg_df["speaker"] = (
    seg_df["speaker"]
    .replace("", np.nan)
    .fillna(seg_df["json_speaker"])
    .fillna(seg_df["json_president"])
)

seg_df["party"] = (
    seg_df["party"]
    .replace("", np.nan)
    .fillna(seg_df["json_party"])
)

seg_df["document_type"] = (
    seg_df["document_type"]
    .replace("", np.nan)
    .fillna(seg_df["json_document_type"])
)

seg_df = seg_df.drop(
    columns=[
        "json_president",
        "json_speaker",
        "json_party",
        "json_document_type",
    ],
    errors="ignore",
)

print("Missing speaker after JSON merge:", seg_df["speaker"].isna().sum())
display(seg_df[["doc_idx", "segment_idx", "speaker", "party", "title", "document_date", "document_type"]].head(20))

# ===================================================
# 4. Corrected segment-level PC1 / PC5 hesapla
# ===================================================

X = seg_df[MFD_COLS].copy()

for c in MFD_COLS:
    X[c] = (X[c] - pca_mean[c]) / pca_scale[c]

Z = X[MFD_COLS].to_numpy(dtype=float)

seg_df["seg_PC1_corrected"] = Z @ pca_loadings["PC1"]
seg_df["seg_PC5_corrected"] = Z @ pca_loadings["PC5"]

seg_df["PC1_energy"] = seg_df["seg_PC1_corrected"] ** 2
seg_df["PC5_energy"] = seg_df["seg_PC5_corrected"] ** 2


# ===================================================
# 5. Belge düzeyinde segment integral enerji üret
# ===================================================

doc_group_cols = [
    "doc_idx",
    "speaker",
    "party",
    "title",
    "document_date",
    "document_type",
]

doc_energy = (
    seg_df
    .groupby(doc_group_cols, dropna=False, as_index=False)
    .agg(
        n_segments=("seg_PC1_corrected", "count"),
        total_word_count=("word_count", "sum"),
        total_sentence_count=("sentence_count", "sum"),
        total_char_count=("char_count", "sum"),

        PC1_integral_energy=("PC1_energy", "sum"),
        PC5_integral_energy=("PC5_energy", "sum"),

        PC1_mean_score=("seg_PC1_corrected", "mean"),
        PC5_mean_score=("seg_PC5_corrected", "mean"),

        PC1_mean_abs_score=("seg_PC1_corrected", lambda x: np.mean(np.abs(x))),
        PC5_mean_abs_score=("seg_PC5_corrected", lambda x: np.mean(np.abs(x))),

        PC1_std_score=("seg_PC1_corrected", "std"),
        PC5_std_score=("seg_PC5_corrected", "std"),
    )
)


# ===================================================
# 6. İstenen metrikler
# ===================================================

doc_energy["PC1_energy_per_1000_words"] = (
    1000 * doc_energy["PC1_integral_energy"] /
    doc_energy["total_word_count"].replace(0, np.nan)
)

doc_energy["PC5_energy_per_1000_words"] = (
    1000 * doc_energy["PC5_integral_energy"] /
    doc_energy["total_word_count"].replace(0, np.nan)
)

doc_energy["PC5_PC1_ratio"] = (
    doc_energy["PC5_energy_per_1000_words"] /
    (doc_energy["PC1_energy_per_1000_words"] + EPS)
)

doc_energy["PC1_PC5_total_energy_per_1000_words"] = (
    doc_energy["PC1_energy_per_1000_words"] +
    doc_energy["PC5_energy_per_1000_words"]
)

doc_energy["PC5_minus_PC1_contrast_per_1000_words"] = (
    doc_energy["PC5_energy_per_1000_words"] -
    doc_energy["PC1_energy_per_1000_words"]
)


# ===================================================
# 7. Ek metrikler
# ===================================================

doc_energy["PC1_energy_share_of_PC1_PC5"] = (
    doc_energy["PC1_energy_per_1000_words"] /
    (doc_energy["PC1_PC5_total_energy_per_1000_words"] + EPS)
)

doc_energy["PC5_energy_share_of_PC1_PC5"] = (
    doc_energy["PC5_energy_per_1000_words"] /
    (doc_energy["PC1_PC5_total_energy_per_1000_words"] + EPS)
)

doc_energy["PC1_rms"] = np.sqrt(
    doc_energy["PC1_integral_energy"] /
    doc_energy["n_segments"].replace(0, np.nan)
)

doc_energy["PC5_rms"] = np.sqrt(
    doc_energy["PC5_integral_energy"] /
    doc_energy["n_segments"].replace(0, np.nan)
)


# ===================================================
# 8. Kaydet — document level
# ===================================================

doc_energy = doc_energy.sort_values(
    ["speaker", "document_date", "doc_idx"],
    na_position="last",
)

doc_out = OUT_DIR / "corpus_pc1_pc5_energy_metrics_segment_level_by_document.csv"

doc_energy.to_csv(
    doc_out,
    index=False,
    encoding="utf-8-sig",
)

display(doc_energy.head())

print("Saved document-level:", doc_out.resolve())


# ===================================================
# 9. Speaker bazında özet
# ===================================================

speaker_energy = (
    doc_energy
    .groupby("speaker", dropna=False, as_index=False)
    .agg(
        n_documents=("doc_idx", "count"),
        total_word_count=("total_word_count", "sum"),
        total_segments=("n_segments", "sum"),

        mean_PC1_energy_per_1000_words=("PC1_energy_per_1000_words", "mean"),
        median_PC1_energy_per_1000_words=("PC1_energy_per_1000_words", "median"),

        mean_PC5_energy_per_1000_words=("PC5_energy_per_1000_words", "mean"),
        median_PC5_energy_per_1000_words=("PC5_energy_per_1000_words", "median"),

        mean_PC5_PC1_ratio=("PC5_PC1_ratio", "mean"),
        median_PC5_PC1_ratio=("PC5_PC1_ratio", "median"),

        mean_PC1_PC5_total_energy_per_1000_words=(
            "PC1_PC5_total_energy_per_1000_words",
            "mean",
        ),
        median_PC1_PC5_total_energy_per_1000_words=(
            "PC1_PC5_total_energy_per_1000_words",
            "median",
        ),

        mean_PC5_minus_PC1_contrast_per_1000_words=(
            "PC5_minus_PC1_contrast_per_1000_words",
            "mean",
        ),
        median_PC5_minus_PC1_contrast_per_1000_words=(
            "PC5_minus_PC1_contrast_per_1000_words",
            "median",
        ),

        mean_PC1_rms=("PC1_rms", "mean"),
        mean_PC5_rms=("PC5_rms", "mean"),
    )
    .sort_values(
        "mean_PC1_PC5_total_energy_per_1000_words",
        ascending=False,
    )
)

speaker_out = OUT_DIR / "corpus_pc1_pc5_energy_metrics_segment_level_by_speaker.csv"

speaker_energy.to_csv(
    speaker_out,
    index=False,
    encoding="utf-8-sig",
)

display(speaker_energy.head())

print("Saved speaker-level:", speaker_out.resolve())


# ===================================================
# 10. Excel export
# ===================================================

excel_out = OUT_DIR / "corpus_pc1_pc5_energy_metrics_segment_level.xlsx"

with pd.ExcelWriter(excel_out, engine="openpyxl") as writer:
    doc_energy.to_excel(writer, sheet_name="by_document", index=False)
    speaker_energy.to_excel(writer, sheet_name="by_speaker", index=False)

print("Saved Excel:", excel_out.resolve())

PCA export loaded.
PC1 loading: [-0.53447322  0.16959786 -0.3807335   0.68310349 -0.27200472]
PC5 loading: [0.48896697 0.36236986 0.46963442 0.61877228 0.16175216]
Segment rows: 289412


,doc_idx,segment_idx,segment_id,speaker,party,title,document_date,document_type,word_count,sentence_count,char_count,care_p,fairness_p,loyalty_p,authority_p,sanctity_p,content
0,0,0,doc_0_seg_0,NaN,NaN,Special Message,1865-12-11 00:00:00,NaN,70,3,404,0.019273,0.014612,0.003767,0.042003,0.029567,To the Senate and House of Representatives of ...
1,1,0,doc_1_seg_0,NaN,NaN,Special Message,1865-12-13 00:00:00,NaN,58,2,321,0.000000,0.054750,0.000317,0.037792,0.000000,To the Senate of the United States: In answer ...
2,2,0,doc_2_seg_0,NaN,NaN,Special Message,1865-12-14 00:00:00,NaN,57,2,358,0.004181,0.030254,0.014815,0.045742,0.019608,To the House of Representatives: In answer to ...
3,3,0,doc_3_seg_0,NaN,NaN,Special Message,1865-12-18 00:00:00,NaN,57,2,356,0.040309,0.042440,0.021485,0.024107,0.000000,To the Senate and House of Representatives of ...
4,4,0,doc_4_seg_0,NaN,NaN,Special Message,1865-12-18 00:00:00,NaN,178,2,1104,0.014317,0.025461,0.025906,0.064271,0.003397,To the Senate of the United States: In reply t...


Doc metadata matched: 53519 / 57427
Missing speaker after JSON merge: 12823


,doc_idx,segment_idx,speaker,party,title,document_date,document_type
0,0,0,Andrew Johnson,National Union,Special Message,1865-12-11 00:00:00,Written
1,1,0,Andrew Johnson,National Union,Special Message,1865-12-13 00:00:00,Written
2,2,0,Andrew Johnson,National Union,Special Message,1865-12-14 00:00:00,Written
3,3,0,Andrew Johnson,National Union,Special Message,1865-12-18 00:00:00,Written
4,4,0,Andrew Johnson,National Union,Special Message,1865-12-18 00:00:00,Written
5,4,1,Andrew Johnson,National Union,Special Message,1865-12-18 00:00:00,Written
6,4,2,Andrew Johnson,National Union,Special Message,1865-12-18 00:00:00,Written
7,4,3,Andrew Johnson,National Union,Special Message,1865-12-18 00:00:00,Written
8,5,0,Andrew Johnson,National Union,Special Message,1865-12-20 00:00:00,Written
9,6,0,Andrew Johnson,National Union,Special Message,1865-12-21 00:00:00,Written


,doc_idx,speaker,party,title,document_date,document_type,n_segments,total_word_count,total_sentence_count,total_char_count,...,PC5_std_score,PC1_energy_per_1000_words,PC5_energy_per_1000_words,PC5_PC1_ratio,PC1_PC5_total_energy_per_1000_words,PC5_minus_PC1_contrast_per_1000_words,PC1_energy_share_of_PC1_PC5,PC5_energy_share_of_PC1_PC5,PC1_rms,PC5_rms
40910,41671,Abraham Lincoln,Republican,Address at Cooper Union in New York City,1860-02-27 00:00:00,Oral,39,7701,289,44858,...,0.463423,16.707942,3.041804,0.182057,19.749745,-13.666138,0.845983,0.154017,1.816363,0.775009
40908,41669,Abraham Lincoln,Republican,Inaugural Address,1861-03-04 00:00:00,Oral,18,3621,132,20987,...,0.393907,6.449950,2.181897,0.338281,8.631847,-4.268053,0.747227,0.252773,1.139085,0.662514
40543,41261,Abraham Lincoln,Republican,Message to the Senate on Royal Arbitration of ...,1861-03-16 00:00:00,Written,1,205,4,1183,...,NaN,18.576564,1.208094,0.065033,19.784657,-17.368470,0.938938,0.061062,1.951460,0.497654
40544,41262,Abraham Lincoln,Republican,Special Message,1861-03-26 00:00:00,Written,1,95,3,545,...,NaN,21.529959,4.920545,0.228544,26.450504,-16.609414,0.813972,0.186028,1.430156,0.683704
40857,41617,Abraham Lincoln,Republican,Proclamation 80—Calling Forth the Militia and ...,1861-04-15 00:00:00,Written,2,410,11,2450,...,0.158189,24.768050,0.186094,0.007513,24.954144,-24.581956,0.992543,0.007457,2.253320,0.195318


Saved document-level: C:\Users\yigit\Desktop\Jedi Order\enlistment data\us_political_corpus\corpus_pc_energy_metrics_segment_level\corpus_pc1_pc5_energy_metrics_segment_level_by_document.csv


,speaker,n_documents,total_word_count,total_segments,mean_PC1_energy_per_1000_words,median_PC1_energy_per_1000_words,mean_PC5_energy_per_1000_words,median_PC5_energy_per_1000_words,mean_PC5_PC1_ratio,median_PC5_PC1_ratio,mean_PC1_PC5_total_energy_per_1000_words,median_PC1_PC5_total_energy_per_1000_words,mean_PC5_minus_PC1_contrast_per_1000_words,median_PC5_minus_PC1_contrast_per_1000_words,mean_PC1_rms,mean_PC5_rms
17,James A. Garfield,2,3032,16,121.928907,121.928907,7.595387,7.595387,0.293525,0.293525,129.524294,129.524294,-114.333519,-114.333519,2.430752,0.809713
2,Andrew Johnson,402,170646,1066,58.985936,39.834186,14.438361,6.431794,8.620520,0.169229,73.424298,51.023360,-44.547575,-28.024211,2.014215,0.855705
9,Franklin Pierce,255,100322,660,54.127285,34.033907,17.203998,8.202096,431.179456,0.240665,71.331283,53.888847,-36.923287,-18.345200,1.765295,0.916796
42,Zachary Taylor,45,20842,126,56.737450,32.471047,10.873011,4.491941,0.541180,0.157184,67.610460,43.349648,-45.864439,-24.099433,2.116664,0.811664
25,John Quincy Adams,168,60689,394,45.596676,32.522978,21.603518,15.007510,4.555915,0.531839,67.200193,51.929437,-23.993158,-11.325603,1.728039,1.144804


Saved speaker-level: C:\Users\yigit\Desktop\Jedi Order\enlistment data\us_political_corpus\corpus_pc_energy_metrics_segment_level\corpus_pc1_pc5_energy_metrics_segment_level_by_speaker.csv
Saved Excel: C:\Users\yigit\Desktop\Jedi Order\enlistment data\us_political_corpus\corpus_pc_energy_metrics_segment_level\corpus_pc1_pc5_energy_metrics_segment_level.xlsx


In [8]:
SEGMENT_CSV_PATH = Path(
    r"C:\Users\yigit\Desktop\Jedi Order\enlistment data\us_political_corpus\processed\presidency_segments_emfd_projection.csv"
)

JSON_PATH = Path(
    r"C:\Users\yigit\Desktop\Jedi Order\enlistment data\us_political_corpus\processed\presidency_enriched_emfd_rhythm.json"
)

PCA_EXPORT_PATH = Path(
    r"C:\Users\yigit\Desktop\Jedi Order\enlistment data\us_political_corpus\pca_export\pca_export_summary.xlsx"
)

In [9]:
sample = pd.read_csv(SEGMENT_CSV_PATH, nrows=5)

print(sample.columns.tolist())

['doc_idx', 'segment_idx', 'segment_id', 'speaker', 'party', 'title', 'document_date', 'document_type', 'word_count', 'sentence_count', 'char_count', 'care_p', 'fairness_p', 'loyalty_p', 'authority_p', 'sanctity_p', 'care_sent', 'fairness_sent', 'loyalty_sent', 'authority_sent', 'sanctity_sent', 'moral_nonmoral_ratio', 'f_var', 'sent_var', 'gonye', 'pergel', 'yasa', 'delta', 'content']


In [13]:
# =====================================================
# Speaker eşleşmeyen belgeler
# =====================================================

missing_docs = (
    seg_df[seg_df["speaker"].isna()]
    .groupby(
        [
            "doc_idx",
            "title",
            "document_date",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        n_segments=("segment_idx", "count"),
        total_words=("word_count", "sum"),
    )
    .sort_values("document_date")
)

display(missing_docs)

missing_docs.to_csv(
    OUT_DIR / "missing_speaker_documents.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Missing documents:", len(missing_docs))

,doc_idx,title,document_date,n_segments,total_words
320,3888,Message in Reply to the House of Representatives,1789-05-08 00:00:00,3,586
321,3889,Message in Reply to the Senate,1789-05-18 00:00:00,5,838
322,3907,Message in Reply to the Senate,1790-01-14 00:00:00,4,641
323,3908,Message in Reply to the House of Representatives,1790-01-14 00:00:00,3,593
324,3930,Message in Reply to the Senate,1790-12-13 00:00:00,3,571
...,...,...,...,...,...
2158,31929,Letter to Congressional Leaders on the Continu...,2017-01-13 00:00:00,1,202
2157,31928,Letter to Congressional Leaders on the Continu...,2017-01-13 00:00:00,3,577
2156,31927,Letter to Congressional Leaders on the Continu...,2017-01-13 00:00:00,1,209
2154,31925,Letter to Congressional Leaders on the Continu...,2017-01-13 00:00:00,1,206


Missing documents: 3908


In [14]:
# =====================================================
# Hangi yıllarda eksik?
# =====================================================

missing_docs["year"] = (
    pd.to_datetime(
        missing_docs["document_date"],
        errors="coerce"
    ).dt.year
)

year_summary = (
    missing_docs
    .groupby("year", as_index=False)
    .agg(
        n_documents=("doc_idx", "count"),
        total_words=("total_words", "sum"),
    )
    .sort_values("year")
)

display(year_summary)

year_summary.to_csv(
    OUT_DIR / "missing_documents_by_year.csv",
    index=False,
    encoding="utf-8-sig",
)

,year,n_documents,total_words
0,1789,2,1424
1,1790,4,2740
2,1791,2,541
3,1792,1,848
4,1793,2,1248
...,...,...,...
169,2013,39,15015
170,2014,72,38746
171,2015,45,15712
172,2016,47,15907


In [15]:
# =====================================================
# 1950 sonrası eksikler
# =====================================================

missing_after_1950 = (
    missing_docs[
        missing_docs["year"] >= 1950
    ]
    .sort_values("document_date")
)

display(missing_after_1950)

missing_after_1950.to_csv(
    OUT_DIR / "missing_documents_after_1950.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Missing documents after 1950:", len(missing_after_1950))

,doc_idx,title,document_date,n_segments,total_words,year
1070,16036,33 - Letter to the Vice President Urging a St...,1950-02-09 00:00:00,6,1092,1950
867,14111,83 - Special Message to the Congress Upon App...,1950-04-03 00:00:00,8,1534,1950
971,14897,Executive Order 10121—Transferring to the Terr...,1950-04-12 00:00:00,6,1467,1950
894,14220,88 - Veto of Bill To Amend the Natural Gas Ac...,1950-04-15 00:00:00,4,824,1950
999,15445,Proclamation 2885—Copyright: Israel,1950-05-04 00:00:00,2,278,1950
...,...,...,...,...,...,...
2154,31925,Letter to Congressional Leaders on the Continu...,2017-01-13 00:00:00,1,206,2017
2160,31932,Letter to Congressional Leaders on the Continu...,2017-01-13 00:00:00,2,353,2017
2155,31926,Letter to Congressional Leaders on the Continu...,2017-01-13 00:00:00,1,203,2017
2158,31929,Letter to Congressional Leaders on the Continu...,2017-01-13 00:00:00,1,202,2017


Missing documents after 1950: 3209


In [16]:
# =====================================================
# Bu belgeler JSON'da gerçekten var mı?
# =====================================================

for _, row in missing_after_1950.head(30).iterrows():
    print("-" * 80)
    print(row["document_date"])
    print(row["title"])

--------------------------------------------------------------------------------
1950-02-09 00:00:00
33  - Letter to the Vice President Urging a Study of the Land and Water Resources of the New England States and New York.
--------------------------------------------------------------------------------
1950-04-03 00:00:00
83  - Special Message to the Congress Upon Approving Bill Relating to Cotton and Peanut Acreage Allotments and Marketing Quotas.
--------------------------------------------------------------------------------
1950-04-12 00:00:00
Executive Order 10121—Transferring to the Territory of Hawaii Title to Certain Public Land Needed for Operation of the Honolulu Airport
--------------------------------------------------------------------------------
1950-04-15 00:00:00
88  - Veto of Bill To Amend the Natural Gas Act of 1938.
--------------------------------------------------------------------------------
1950-05-04 00:00:00
Proclamation 2885—Copyright: Israel
---------------

In [18]:
# BLOK — Segment-level corrected PCA dosyası üret + JSON metadata ile speaker doldur

from pathlib import Path
import json
import re
import numpy as np
import pandas as pd


# ===================================================
# 1. Ayarlar
# ===================================================

SEGMENT_CSV_PATH = Path(
    r"C:\Users\yigit\Desktop\Jedi Order\enlistment data\us_political_corpus\processed\presidency_segments_emfd_projection.csv"
)

JSON_PATH = Path(
    r"C:\Users\yigit\Desktop\Jedi Order\enlistment data\us_political_corpus\processed\presidency_enriched_emfd_rhythm.json"
)

PCA_EXPORT_PATH = Path(
    r"C:\Users\yigit\Desktop\Jedi Order\enlistment data\us_political_corpus\pca_export\pca_export_summary.xlsx"
)

OUT_DIR = Path(
    r"C:\Users\yigit\Desktop\Jedi Order\enlistment data\us_political_corpus\processed"
)

OUT_CSV = OUT_DIR / "presidency_segments_corrected_pca.csv"
OUT_PARQUET = OUT_DIR / "presidency_segments_corrected_pca.parquet"

MFD_COLS = [
    "care_p",
    "fairness_p",
    "loyalty_p",
    "authority_p",
    "sanctity_p",
]

USECOLS = [
    "doc_idx",
    "segment_idx",
    "segment_id",
    "speaker",
    "party",
    "title",
    "document_date",
    "document_type",
    "word_count",
    "sentence_count",
    "char_count",
    "care_p",
    "fairness_p",
    "loyalty_p",
    "authority_p",
    "sanctity_p",
    "care_sent",
    "fairness_sent",
    "loyalty_sent",
    "authority_sent",
    "sanctity_sent",
    "moral_nonmoral_ratio",
    "f_var",
    "sent_var",
    "gonye",
    "pergel",
    "yasa",
    "delta",
    "content",
]


# ===================================================
# 2. PCA export oku
# ===================================================

def load_pca_export(path):
    xls = pd.ExcelFile(path)

    scaler_stats = pd.read_excel(path, sheet_name="scaler_stats")

    if "loading_matrix_aligned" in xls.sheet_names:
        loadings = pd.read_excel(path, sheet_name="loading_matrix_aligned")
    else:
        loadings = pd.read_excel(path, sheet_name="loading_matrix")

    mean = scaler_stats.set_index("feature")["mean"].to_dict()
    scale = scaler_stats.set_index("feature")["scale"].to_dict()

    loading_matrix = {}

    for pc in ["PC1", "PC2", "PC3", "PC4", "PC5"]:
        loading_matrix[pc] = (
            loadings
            .set_index("feature")
            .loc[MFD_COLS, pc]
            .to_numpy(dtype=float)
        )

    return mean, scale, loading_matrix


pca_mean, pca_scale, pca_loadings = load_pca_export(PCA_EXPORT_PATH)

print("PCA export loaded.")
print("PC1 loading:", pca_loadings["PC1"])
print("PC5 loading:", pca_loadings["PC5"])


# ===================================================
# 3. Segment CSV oku
# ===================================================

sample = pd.read_csv(SEGMENT_CSV_PATH, nrows=5)
available_cols = sample.columns.tolist()

usecols = [c for c in USECOLS if c in available_cols]

missing_cols = [c for c in USECOLS if c not in available_cols]
if missing_cols:
    print("Missing columns skipped:", missing_cols)

seg_df = pd.read_csv(
    SEGMENT_CSV_PATH,
    usecols=usecols,
)

print("Loaded segment rows:", len(seg_df))
print("Loaded columns:", len(seg_df.columns))


# ===================================================
# 4. JSON metadata ile speaker/party/document_type doldur
# Merge: title + date
# ===================================================

def norm_text(x):
    if pd.isna(x):
        return ""
    x = str(x).lower()
    x = re.sub(r"\s+", " ", x)
    return x.strip()


def norm_date_series(s):
    return pd.to_datetime(s, errors="coerce").dt.strftime("%Y-%m-%d")


with open(JSON_PATH, "r", encoding="utf-8") as f:
    meta = json.load(f)

json_rows = []

for president_name, pdata in meta.items():

    if not isinstance(pdata, dict):
        continue

    for doc_type_key, docs in pdata.items():

        if not isinstance(docs, list):
            continue

        for doc in docs:

            if not isinstance(doc, dict):
                continue

            json_rows.append({
                "json_president": president_name,
                "json_speaker": doc.get("speaker", president_name),
                "json_party": doc.get("party", None),
                "json_title": doc.get("title", None),
                "json_document_date": doc.get("document_date", None),
                "json_document_type": doc.get("document_type", doc_type_key),
            })

json_meta_df = pd.DataFrame(json_rows)

json_meta_df["merge_title"] = json_meta_df["json_title"].apply(norm_text)
json_meta_df["merge_date"] = norm_date_series(json_meta_df["json_document_date"])

seg_doc_keys = (
    seg_df
    .groupby("doc_idx", as_index=False)
    .agg(
        merge_title=("title", lambda x: norm_text(x.iloc[0])),
        merge_date=("document_date", lambda x: pd.to_datetime(x.iloc[0], errors="coerce").strftime("%Y-%m-%d")),
    )
)

json_map = (
    json_meta_df[
        [
            "merge_title",
            "merge_date",
            "json_president",
            "json_speaker",
            "json_party",
            "json_document_type",
        ]
    ]
    .drop_duplicates(["merge_title", "merge_date"])
)

doc_meta_map = seg_doc_keys.merge(
    json_map,
    on=["merge_title", "merge_date"],
    how="left",
)

print("Doc metadata matched:", doc_meta_map["json_speaker"].notna().sum(), "/", len(doc_meta_map))

seg_df = seg_df.merge(
    doc_meta_map[
        [
            "doc_idx",
            "json_president",
            "json_speaker",
            "json_party",
            "json_document_type",
        ]
    ],
    on="doc_idx",
    how="left",
)

seg_df["speaker"] = (
    seg_df["speaker"]
    .replace("", np.nan)
    .fillna(seg_df["json_speaker"])
    .fillna(seg_df["json_president"])
)

seg_df["party"] = (
    seg_df["party"]
    .replace("", np.nan)
    .fillna(seg_df["json_party"])
)

seg_df["document_type"] = (
    seg_df["document_type"]
    .replace("", np.nan)
    .fillna(seg_df["json_document_type"])
)

seg_df = seg_df.drop(
    columns=[
        "json_president",
        "json_speaker",
        "json_party",
        "json_document_type",
    ],
    errors="ignore",
)

print("Missing speaker after JSON merge:", seg_df["speaker"].isna().sum())


# ===================================================
# 5. Numerik kolonları temizle
# ===================================================

for c in MFD_COLS:
    seg_df[c] = pd.to_numeric(seg_df[c], errors="coerce").fillna(0)

numeric_cols = [
    "doc_idx",
    "segment_idx",
    "word_count",
    "sentence_count",
    "char_count",
    "care_sent",
    "fairness_sent",
    "loyalty_sent",
    "authority_sent",
    "sanctity_sent",
    "moral_nonmoral_ratio",
    "f_var",
    "sent_var",
    "gonye",
    "pergel",
    "yasa",
    "delta",
]

for c in numeric_cols:
    if c in seg_df.columns:
        seg_df[c] = pd.to_numeric(seg_df[c], errors="coerce")


# ===================================================
# 6. Corrected PCA skorlarını hesapla
# ===================================================

X = seg_df[MFD_COLS].copy()

for c in MFD_COLS:
    X[c] = (X[c] - pca_mean[c]) / pca_scale[c]

Z = X[MFD_COLS].to_numpy(dtype=float)

for pc in ["PC1", "PC2", "PC3", "PC4", "PC5"]:
    seg_df[f"seg_{pc}_corrected"] = Z @ pca_loadings[pc]


# ===================================================
# 7. Enerji kolonları
# ===================================================

for pc in ["PC1", "PC2", "PC3", "PC4", "PC5"]:
    seg_df[f"{pc}_energy"] = seg_df[f"seg_{pc}_corrected"] ** 2

seg_df["PC1_PC5_total_energy"] = (
    seg_df["PC1_energy"] + seg_df["PC5_energy"]
)

seg_df["PC5_minus_PC1_energy"] = (
    seg_df["PC5_energy"] - seg_df["PC1_energy"]
)

seg_df["PC5_PC1_energy_ratio"] = (
    seg_df["PC5_energy"] / (seg_df["PC1_energy"] + 1e-12)
)


# ===================================================
# 8. Kolon sırası
# ===================================================

preferred_order = [
    "doc_idx",
    "segment_idx",
    "segment_id",
    "speaker",
    "party",
    "title",
    "document_date",
    "document_type",
    "word_count",
    "sentence_count",
    "char_count",
    *MFD_COLS,
    "seg_PC1_corrected",
    "seg_PC2_corrected",
    "seg_PC3_corrected",
    "seg_PC4_corrected",
    "seg_PC5_corrected",
    "PC1_energy",
    "PC2_energy",
    "PC3_energy",
    "PC4_energy",
    "PC5_energy",
    "PC1_PC5_total_energy",
    "PC5_minus_PC1_energy",
    "PC5_PC1_energy_ratio",
    "care_sent",
    "fairness_sent",
    "loyalty_sent",
    "authority_sent",
    "sanctity_sent",
    "moral_nonmoral_ratio",
    "f_var",
    "sent_var",
    "gonye",
    "pergel",
    "yasa",
    "delta",
    "content",
]

ordered_cols = [c for c in preferred_order if c in seg_df.columns]
remaining_cols = [c for c in seg_df.columns if c not in ordered_cols]

seg_df = seg_df[ordered_cols + remaining_cols]


# ===================================================
# 9. Kaydet
# ===================================================

seg_df.to_csv(
    OUT_CSV,
    index=False,
    encoding="utf-8-sig",
)

print("Saved CSV:")
print(OUT_CSV.resolve())

try:
    seg_df.to_parquet(
        OUT_PARQUET,
        index=False,
    )

    print("Saved Parquet:")
    print(OUT_PARQUET.resolve())

except Exception as e:
    print("Parquet kaydedilemedi.")
    print("Hata:", e)


# ===================================================
# 10. Kontrol
# ===================================================

display(
    seg_df[
        [
            "doc_idx",
            "segment_idx",
            "speaker",
            "party",
            "title",
            "document_date",
            "document_type",
            "seg_PC1_corrected",
            "seg_PC5_corrected",
        ]
    ].head(20)
)

print("Rows:", len(seg_df))
print("Columns:", len(seg_df.columns))
print("Missing speaker:", seg_df["speaker"].isna().sum())

PCA export loaded.
PC1 loading: [-0.53447322  0.16959786 -0.3807335   0.68310349 -0.27200472]
PC5 loading: [0.48896697 0.36236986 0.46963442 0.61877228 0.16175216]
Loaded segment rows: 289412
Loaded columns: 29
Doc metadata matched: 57427 / 57427
Missing speaker after JSON merge: 0
Saved CSV:
C:\Users\yigit\Desktop\Jedi Order\enlistment data\us_political_corpus\processed\presidency_segments_corrected_pca.csv
Saved Parquet:
C:\Users\yigit\Desktop\Jedi Order\enlistment data\us_political_corpus\processed\presidency_segments_corrected_pca.parquet


,doc_idx,segment_idx,speaker,party,title,document_date,document_type,seg_PC1_corrected,seg_PC5_corrected
0,0,0,Andrew Johnson,National Union,Special Message,1865-12-11 00:00:00,Written,0.979626,-1.074015
1,1,0,Andrew Johnson,National Union,Special Message,1865-12-13 00:00:00,Written,3.028783,-1.549805
2,2,0,Andrew Johnson,National Union,Special Message,1865-12-14 00:00:00,Written,1.793531,-0.714409
3,3,0,Andrew Johnson,National Union,Special Message,1865-12-18 00:00:00,Written,0.320731,-0.431040
4,4,0,Andrew Johnson,National Union,Special Message,1865-12-18 00:00:00,Written,2.486980,0.320752
5,4,1,Andrew Johnson,National Union,Special Message,1865-12-18 00:00:00,Written,1.016123,0.213614
6,4,2,Andrew Johnson,National Union,Special Message,1865-12-18 00:00:00,Written,0.252485,0.639837
7,4,3,Andrew Johnson,National Union,Special Message,1865-12-18 00:00:00,Written,3.030563,0.279596
8,5,0,Andrew Johnson,National Union,Special Message,1865-12-20 00:00:00,Written,1.175727,-0.868132
9,6,0,Andrew Johnson,National Union,Special Message,1865-12-21 00:00:00,Written,6.265689,0.940386


Rows: 289412
Columns: 42
Missing speaker: 0


In [19]:
# BLOK — Master feature table kolonlarını ekle ve ayrı dosyaya kaydet
# Girdi:
# presidency_segments_corrected_pca.csv
#
# Çıktı:
# presidency_segments_corrected_pca_master_features.csv
# presidency_segments_corrected_pca_master_features.parquet

from pathlib import Path
import numpy as np
import pandas as pd


# ===================================================
# 1. Ayarlar
# ===================================================

IN_PATH = Path(
    r"C:\Users\yigit\Desktop\Jedi Order\enlistment data\us_political_corpus\processed\presidency_segments_corrected_pca.csv"
)

OUT_DIR = IN_PATH.parent

OUT_CSV = OUT_DIR / "presidency_segments_corrected_pca_master_features.csv"
OUT_PARQUET = OUT_DIR / "presidency_segments_corrected_pca_master_features.parquet"

EPS = 1e-12


# ===================================================
# 2. Dosyayı oku
# ===================================================

df = pd.read_csv(IN_PATH)

print("Loaded rows:", len(df))
print("Loaded columns:", len(df.columns))


# ===================================================
# 3. Numerik kolonları garantiye al
# ===================================================

numeric_cols = [
    "doc_idx",
    "segment_idx",
    "word_count",
    "seg_PC1_corrected",
    "seg_PC2_corrected",
    "seg_PC3_corrected",
    "seg_PC4_corrected",
    "seg_PC5_corrected",
]

for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")


# ===================================================
# 4. Document-level yardımcı değerler
# ===================================================

df["doc_segment_count"] = (
    df.groupby("doc_idx")["segment_idx"]
    .transform("count")
)

df["total_document_word_count"] = (
    df.groupby("doc_idx")["word_count"]
    .transform("sum")
)


# ===================================================
# 5. segment_order_norm
# ===================================================

df["segment_order_norm"] = np.where(
    df["doc_segment_count"] > 1,
    df["segment_idx"] / (df["doc_segment_count"] - 1),
    0.0,
)


# ===================================================
# 6. PC energy kolonları
# ===================================================

for pc in ["PC1", "PC2", "PC3", "PC4", "PC5"]:
    score_col = f"seg_{pc}_corrected"
    energy_col = f"{pc}_energy"

    df[energy_col] = df[score_col] ** 2


# ===================================================
# 7. GPY enerji payı
# ===================================================

df["PC1345_total_energy"] = (
    df["PC1_energy"]
    + df["PC3_energy"]
    + df["PC4_energy"]
    + df["PC5_energy"]
)

df["GPY_energy_share"] = (
    df["PC5_energy"] /
    (df["PC1345_total_energy"] + EPS)
)


# ===================================================
# 8. PC1-PC5 angle
# ===================================================
# Burada angle, segmentin (PC1, PC5) düzlemindeki yön açısıdır.
# atan2(y, x) = atan2(PC5, PC1)
# Radyan ve derece olarak kaydedilir.

df["PC1_PC5_angle_rad"] = np.arctan2(
    df["seg_PC5_corrected"],
    df["seg_PC1_corrected"],
)

df["PC1_PC5_angle_deg"] = np.degrees(
    df["PC1_PC5_angle_rad"]
)


# ===================================================
# 9. segment_word_fraction
# ===================================================

df["segment_word_fraction"] = (
    df["word_count"] /
    (df["total_document_word_count"] + EPS)
)


# ===================================================
# 10. Ek pratik PC1-PC5 metrikleri
# ===================================================

df["PC1_PC5_total_energy"] = (
    df["PC1_energy"] + df["PC5_energy"]
)

df["PC5_minus_PC1_energy"] = (
    df["PC5_energy"] - df["PC1_energy"]
)

df["PC5_PC1_energy_ratio"] = (
    df["PC5_energy"] /
    (df["PC1_energy"] + EPS)
)

df["PC5_energy_share_of_PC1_PC5"] = (
    df["PC5_energy"] /
    (df["PC1_PC5_total_energy"] + EPS)
)

df["PC1_energy_share_of_PC1_PC5"] = (
    df["PC1_energy"] /
    (df["PC1_PC5_total_energy"] + EPS)
)


# ===================================================
# 11. Kolon sırası
# ===================================================

preferred_order = [
    "doc_idx",
    "segment_idx",
    "segment_id",
    "speaker",
    "party",
    "title",
    "document_date",
    "document_type",

    "doc_segment_count",
    "segment_order_norm",
    "word_count",
    "total_document_word_count",
    "segment_word_fraction",
    "sentence_count",
    "char_count",

    "care_p",
    "fairness_p",
    "loyalty_p",
    "authority_p",
    "sanctity_p",

    "seg_PC1_corrected",
    "seg_PC2_corrected",
    "seg_PC3_corrected",
    "seg_PC4_corrected",
    "seg_PC5_corrected",

    "PC1_energy",
    "PC2_energy",
    "PC3_energy",
    "PC4_energy",
    "PC5_energy",

    "PC1345_total_energy",
    "GPY_energy_share",

    "PC1_PC5_angle_rad",
    "PC1_PC5_angle_deg",

    "PC1_PC5_total_energy",
    "PC5_minus_PC1_energy",
    "PC5_PC1_energy_ratio",
    "PC5_energy_share_of_PC1_PC5",
    "PC1_energy_share_of_PC1_PC5",

    "care_sent",
    "fairness_sent",
    "loyalty_sent",
    "authority_sent",
    "sanctity_sent",

    "moral_nonmoral_ratio",
    "f_var",
    "sent_var",

    "gonye",
    "pergel",
    "yasa",
    "delta",

    "content",
]

ordered_cols = [c for c in preferred_order if c in df.columns]
remaining_cols = [c for c in df.columns if c not in ordered_cols]

df = df[ordered_cols + remaining_cols]


# ===================================================
# 12. Kaydet
# ===================================================

df.to_csv(
    OUT_CSV,
    index=False,
    encoding="utf-8-sig",
)

print("Saved CSV:")
print(OUT_CSV.resolve())

try:
    df.to_parquet(
        OUT_PARQUET,
        index=False,
    )

    print("Saved Parquet:")
    print(OUT_PARQUET.resolve())

except Exception as e:
    print("Parquet kaydedilemedi.")
    print("Hata:", e)


# ===================================================
# 13. Kontrol
# ===================================================

display(
    df[
        [
            "doc_idx",
            "segment_idx",
            "speaker",
            "title",
            "segment_order_norm",
            "segment_word_fraction",
            "seg_PC1_corrected",
            "seg_PC5_corrected",
            "PC1_energy",
            "PC5_energy",
            "GPY_energy_share",
            "PC1_PC5_angle_deg",
        ]
    ].head(20)
)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Missing speaker:", df["speaker"].isna().sum())

C:\Users\yigit\AppData\Local\Temp\ipykernel_5520\1146429350.py:34: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(IN_PATH)


Loaded rows: 289412
Loaded columns: 42
Saved CSV:
C:\Users\yigit\Desktop\Jedi Order\enlistment data\us_political_corpus\processed\presidency_segments_corrected_pca_master_features.csv
Saved Parquet:
C:\Users\yigit\Desktop\Jedi Order\enlistment data\us_political_corpus\processed\presidency_segments_corrected_pca_master_features.parquet


,doc_idx,segment_idx,speaker,title,segment_order_norm,segment_word_fraction,seg_PC1_corrected,seg_PC5_corrected,PC1_energy,PC5_energy,GPY_energy_share,PC1_PC5_angle_deg
0,0,0,Andrew Johnson,Special Message,0.000000,1.000000,0.979626,-1.074015,0.959668,1.153507,0.080252,-47.631543
1,1,0,Andrew Johnson,Special Message,0.000000,1.000000,3.028783,-1.549805,9.173525,2.401896,0.187179,-27.098482
2,2,0,Andrew Johnson,Special Message,0.000000,1.000000,1.793531,-0.714409,3.216753,0.510380,0.061456,-21.718652
3,3,0,Andrew Johnson,Special Message,0.000000,1.000000,0.320731,-0.431040,0.102868,0.185795,0.090801,-53.347589
4,4,0,Andrew Johnson,Special Message,0.000000,0.275969,2.486980,0.320752,6.185071,0.102882,0.016270,7.349019
5,4,1,Andrew Johnson,Special Message,0.333333,0.217054,1.016123,0.213614,1.032505,0.045631,0.030130,11.872125
6,4,2,Andrew Johnson,Special Message,0.666667,0.308527,0.252485,0.639837,0.063749,0.409392,0.466588,68.465405
7,4,3,Andrew Johnson,Special Message,1.000000,0.198450,3.030563,0.279596,9.184311,0.078174,0.005367,5.271117
8,5,0,Andrew Johnson,Special Message,0.000000,1.000000,1.175727,-0.868132,1.382334,0.753652,0.255002,-36.441363
9,6,0,Andrew Johnson,Special Message,0.000000,1.000000,6.265689,0.940386,39.258861,0.884326,0.020191,8.535530


Rows: 289412
Columns: 52
Missing speaker: 0
